<a href="https://colab.research.google.com/github/jefferyocran/FraudGuard-OXGBoost/blob/main/experiment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 4 — Decision Threshold Analysis

**Question:** Can scam recall be improved by adjusting the classification threshold, without collecting more data?

Trains a cost-sensitive XGBoost model (scale_pos_weight = legitimate/scam ratio) and evaluates recall and precision across a range of decision thresholds on the Ghanaian test set.

In [1]:
!pip install -U xgboost -q

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import recall_score, precision_score

SEED = 42
np.random.seed(SEED)
print("Ready. Seed =", SEED)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.4/252.4 MB 4.8 MB/s eta 0:00:00
Ready. Seed = 42


In [2]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
uci = pd.read_csv(url, sep='\t', header=None, names=['label','message'])
uci['label'] = uci['label'].map({'ham':0,'spam':1}); uci['source']='uci'
field = pd.read_csv('ghana_momo_field.csv')[['label','message']]; field['source']='field'
print("UCI:", len(uci), "| Field:", len(field))

UCI: 5572 | Field: 208


In [3]:
# Localised training set + held-out Ghanaian test set
f_tr, f_te = train_test_split(field, test_size=0.4, stratify=field['label'], random_state=SEED)
train_B = pd.concat([uci, f_tr], ignore_index=True)

vec = TfidfVectorizer(max_features=1000, ngram_range=(1,2))
Xtr = vec.fit_transform(train_B['message']); ytr = train_B['label']
Xte = vec.transform(f_te['message']);        yte = f_te['label']
print("Train:", Xtr.shape[0], "| Test:", Xte.shape[0])

Train: 5696 | Test: 84


In [4]:
# Principled imbalance weighting: scale_pos_weight = #legit / #scam
n_neg=(ytr==0).sum(); n_pos=(ytr==1).sum(); spw=n_neg/n_pos
print(f"scale_pos_weight = {spw:.2f}")

model = xgb.XGBClassifier(max_depth=6, n_estimators=200, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
    random_state=SEED, eval_metric='logloss')
model.fit(Xtr, ytr)
probs = model.predict_proba(Xte)[:,1]
print("Model trained.")

scale_pos_weight = 6.38
Model trained.


In [5]:
print("Threshold | Recall | Precision | Scams caught")
print("-"*48)
for t in [0.50,0.40,0.30,0.25,0.20,0.15,0.10]:
    preds=(probs>=t).astype(int)
    r=recall_score(yte,preds,pos_label=1,zero_division=0)
    p=precision_score(yte,preds,pos_label=1,zero_division=0)
    caught=int(((preds==1)&(yte==1)).sum()); tot=int((yte==1).sum())
    print(f"   {t:.2f}   | {r*100:5.1f}% |  {p*100:5.1f}%  |  {caught}/{tot}")

Threshold | Recall | Precision | Scams caught
------------------------------------------------
   0.50   |  35.3% |   28.6%  |  6/17
   0.40   |  52.9% |   36.0%  |  9/17
   0.30   |  58.8% |   29.4%  |  10/17
   0.25   |  58.8% |   27.0%  |  10/17
   0.20   |  58.8% |   22.2%  |  10/17
   0.15   |  70.6% |   24.0%  |  12/17
   0.10   |  88.2% |   25.4%  |  15/17
